# Cleaning Pipeline 02 — Exact Duplicate Detection with MD5

An exact duplicate has identical file bytes. MD5 is used as a fast file fingerprint.

**Why this matters:** repeated images can overweight one scene, leak across CV folds, and inflate validation. Cross-label exact duplicates are strong evidence of annotation inconsistency.

**Safe policy:** for a same-label group, keep one file and mark the rest removable. For a cross-label group, review manually.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "cleaning_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))
DATA_ROOT = REPO_ROOT / "BDC2026"
OUTPUT_ROOT = REPO_ROOT / "eda_outputs" / "notebook_pipeline"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LABEL2ID = {"0_Recyclable": 0, "1_Electronic": 1, "2_Organic": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm

manifest_path = OUTPUT_ROOT / "00_manifest" / "train_manifest.csv"
if not manifest_path.exists():
    raise FileNotFoundError(f"Run notebook 00 first: {manifest_path}")
manifest = pd.read_csv(manifest_path)
valid = manifest[manifest["is_valid"]].copy().reset_index(drop=True)

In [ ]:
def file_md5(path: str, chunk_size: int = 1024 * 1024):
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

valid["md5"] = [file_md5(path) for path in tqdm(valid["path"], desc="Computing MD5")]

In [ ]:
duplicate_rows = []
group_id = 0

for signature, group in valid.groupby("md5", sort=False):
    if len(group) <= 1:
        continue
    labels = sorted(group["label"].unique().tolist())
    cross_label = len(labels) > 1
    group = group.sort_values("path").reset_index(drop=True)

    for position, (_, row) in enumerate(group.iterrows()):
        action = "manual_review_cross_label" if cross_label else ("keep" if position == 0 else "remove_duplicate")
        duplicate_rows.append({
            "group_id": group_id,
            "md5": signature,
            "path": row["path"],
            "relative_path": row["relative_path"],
            "class_name": row["class_name"],
            "label": int(row["label"]),
            "cross_label": cross_label,
            "recommended_action": action,
        })
    group_id += 1

exact_columns = ["group_id", "md5", "path", "relative_path", "class_name", "label", "cross_label", "recommended_action"]
exact = pd.DataFrame(duplicate_rows, columns=exact_columns)
print("Duplicate groups:", exact["group_id"].nunique() if len(exact) else 0)
print("Duplicate rows:", len(exact))
print("Cross-label groups:", exact.loc[exact["cross_label"], "group_id"].nunique() if len(exact) else 0)
display(exact.head(30))

In [ ]:
def show_duplicate_group(group):
    cols = min(5, len(group))
    rows = int(np.ceil(len(group) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.5))
    axes = np.asarray(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for ax, (_, row) in zip(axes, group.iterrows()):
        with Image.open(row["path"]) as img:
            ax.imshow(ImageOps.exif_transpose(img).convert("RGB"))
        ax.set_title(f"{Path(row['path']).name}\n{row['class_name']}\n{row['recommended_action']}", fontsize=8)
    plt.tight_layout(); plt.show()

for gid in exact["group_id"].drop_duplicates().head(5):
    print("Group", gid)
    show_duplicate_group(exact[exact["group_id"] == gid])

## CV leakage note

Exact duplicate grouping should happen before cross-validation splitting. All copies of the same image should remain in the same fold, or redundant copies should be removed before folds are created.

In [ ]:
stage_dir = OUTPUT_ROOT / "02_exact_duplicates"
stage_dir.mkdir(parents=True, exist_ok=True)
valid[["path","relative_path","class_name","label","md5"]].to_csv(stage_dir / "image_md5.csv", index=False)
exact.to_csv(stage_dir / "exact_duplicate_groups.csv", index=False)
exact[exact["recommended_action"] == "remove_duplicate"].to_csv(stage_dir / "exact_duplicates_auto_remove.csv", index=False)
exact[exact["recommended_action"] == "manual_review_cross_label"].to_csv(stage_dir / "exact_duplicates_manual_review.csv", index=False)
print("Saved outputs to", stage_dir)